# WRF Tropical Cyclone: Comparing Runs

Compare a WRF variable between the `era5_default` and `slab_ocean` runs.
Change `var` in the settings cell to inspect a different field.


In [ ]:
import os
from pathlib import Path

env_path = Path('.env')
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        k, v = line.split('=', 1)
        os.environ.setdefault(k.strip(), v.strip())

user = os.environ['USERNAME']


In [2]:
# === USER SETTINGS ===
wrfout_pattern = f"/cluster/home/{user}/wcm/wrf-model/wrf-output/era5_default/wrfout_d01_*"
ocean_pattern  = f"/cluster/home/{user}/wcm/wrf-model/wrf-output/slab_ocean/wrfout_d01_*"

# not too interesting only contains a few variables that are not really relevant for our project
obs_file    = f"/cluster/home/{user}/wcm/wrf-model/OBS/matthew_obs.nc"

In [3]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import ipywidgets as widgets
from IPython.display import display

import wrf_plot as wp
from storm_track import storm_track


ModuleNotFoundError: No module named 'tqdm'

In [ ]:
# --- Load WRF runs, observations, and storm track ---
ds_default = wp.load_wrf(wrfout_pattern)
ds_ocean   = wp.load_wrf(ocean_pattern)
# ds_obs     = wp.load_obs(obs_file)

lat, lon = wp.get_latlon(ds_default)

# Storm track from the era5_default run (min-SLP location per time step)
track_pattern = f"/cluster/home/{user}/wcm/wrf-model/wrf-output/era5_default/wrfout_d01_*"
track = storm_track(track_pattern)


In [ ]:
# @title Choose variable and storm-center radius
surface_vars = sorted(
    v for v in ds_default.data_vars
    if ds_default[v].dims == ("Time", "south_north", "west_east")
)
default_var = "slp" if "slp" in surface_vars else surface_vars[0]

plot_var = widgets.Dropdown(
    options=surface_vars, value=default_var, description="Variable"
)
plot_radius = widgets.IntSlider(
    min=0, max=500, step=10, value=100,
    description="Radius (km)", continuous_update=False,
)

display(widgets.VBox([
    plot_var,
    plot_radius,
    widgets.Label("Run the next cells to plot. Rerun after changing values."),
]))


In [ ]:
# @title Plot: era5_default vs slab_ocean (scrub through time)
var = plot_var.value
radius = plot_radius.value or None

data = {
    f"era5_default: {var}": ds_default[var],
    f"slab_ocean: {var}":   ds_ocean[var],
}

wp.plot_time(
    data,
    fig_title=f"{var}",
    cols=3,
    track=track,
    lat=lat,
    lon=lon,
    radius=radius,
    cmap="gray_r",
    vmin=None,
    vmax=None,
    coastlines=True,
    cut_line=True,
)


In [ ]:
# @title Time series at the storm center
var = plot_var.value
radius = plot_radius.value or None

data = {
    f"era5_default: {var}": ds_default[var],
    f"slab_ocean: {var}":   ds_ocean[var],
}

fig, series = wp.plot_track_time(
    data,
    track=track, lat=lat, lon=lon,
    fig_title=f"{var} along storm track", radius=radius,
)
plt.show()


## Vertical cross-section through the storm center
Cut runs parallel to latitudes (constant lat = storm-center lat), with width 2·radius in longitude. Only multi-level variables can be selected.

In [ ]:
# @title Choose multi-level variable for the cross-section
level_vars = sorted(set(wp.multi_level_vars(ds_default)) & set(ds_ocean.data_vars))
default_lvar = "QVAPOR" if "QVAPOR" in level_vars else level_vars[0]

cs_var = widgets.Dropdown(options=level_vars, value=default_lvar, description="Variable")
display(cs_var)

In [ ]:
# @title Plot: vertical cross-section, era5_default vs slab_ocean

# this can take a while

var = cs_var.value
radius = plot_radius.value or 100

data = {
    f"era5_default: {var}": ds_default[var],
    f"slab_ocean: {var}":   ds_ocean[var],
}

wp.plot_cross_section(
    data, track=track, lat=lat, lon=lon, radius=radius,
    fig_title=f"{var} cross-section", cmap="viridis",
)